# CS1-EXP0-LR — Project-Holdout Test + Inner Grouped CV

## Objective

This notebook runs the first Case Study 1 experiment with a two-stage,
leakage-aware evaluation design:

```text
All RDiverseVul functions
│
├── Outer project-disjoint holdout (approximately 20% of rows)
│   └── Reserved for one final test after model selection is complete.
│
└── Development partition (remaining projects)
    └── Inner 5-fold StratifiedGroupKFold by project
        ├── profile computational feasibility
        ├── compare candidate models / configurations
        ├── inspect pooled OOF results
        └── choose the final candidate without using outer-holdout metrics
```

## What this notebook does

- creates and freezes a project-disjoint outer holdout before model development;
- creates a separate fixed 5-fold project-grouped manifest only on development data;
- runs EXP-0 only on the development partition;
- saves all artifacts in experiment-specific folders;
- keeps the final holdout-evaluation cell locked until a model is selected.

## What this notebook does **not** do

- It does not use outer-holdout scores to select a threshold or tune EXP-0.
- It does not overwrite earlier manifests or experiment outputs.
- It does not claim inner-CV performance as final holdout performance.

> The primary model-selection metric is pooled out-of-fold PR-AUC on the
> development partition. Accuracy is retained only as a secondary descriptive metric
> because the data are highly imbalanced.

## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Define immutable experiment paths

In [2]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

RAW_DIR = DRIVE_ROOT / "raw"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

# A versioned experiment root prevents accidental overwrite or mixing of
# full-data CV, within-project CV, and holdout+inner-CV artifacts.
EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"

EXPERIMENT_MANIFEST_DIR = MANIFEST_ROOT / EXPERIMENT_ID
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

SETUP_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "setup"
INNER_CV_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "exp0_inner_cv"
FINAL_HOLDOUT_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "exp0_final_holdout"

OUTER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "outer_holdout"
INNER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "inner_cv"

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_ROOT,
    OUTPUT_ROOT,
    EXPERIMENT_MANIFEST_DIR,
    EXPERIMENT_OUTPUT_DIR,
    SETUP_OUTPUT_DIR,
    INNER_CV_OUTPUT_DIR,
    FINAL_HOLDOUT_OUTPUT_DIR,
    OUTER_SPLIT_DIR,
    INNER_SPLIT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

NORMALIZATION_REPORT_PATH = (
    SETUP_OUTPUT_DIR / "normalization_summary.json"
)

AUDIT_OUTPUT_PATH = SETUP_OUTPUT_DIR / "dataset_audit.json"

OUTER_MANIFEST_PATH = (
    OUTER_SPLIT_DIR / "cs1_outer_project_holdout_manifest.parquet"
)

OUTER_CANDIDATES_PATH = (
    OUTER_SPLIT_DIR / "cs1_outer_holdout_candidates.csv"
)

OUTER_METADATA_PATH = (
    OUTER_SPLIT_DIR / "cs1_outer_project_holdout_metadata.json"
)

INNER_MANIFEST_PATH = (
    INNER_SPLIT_DIR / "cs1_project_grouped_5fold_manifest.parquet"
)

INNER_MANIFEST_SUMMARY_PATH = (
    INNER_SPLIT_DIR / "cs1_project_grouped_5fold_fold_summary.csv"
)

print("Experiment ID:", EXPERIMENT_ID)
print("Normalized cache:", NORMALIZED_DATA_PATH)
print("Outer holdout manifest:", OUTER_MANIFEST_PATH)
print("Inner CV manifest:", INNER_MANIFEST_PATH)
print("Inner CV output directory:", INNER_CV_OUTPUT_DIR)
print("Final holdout output directory:", FINAL_HOLDOUT_OUTPUT_DIR)

Experiment ID: cs1_project_holdout20_innercv_v1
Normalized cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
Outer holdout manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Inner CV manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Inner CV output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp0_inner_cv
Final holdout output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp0_final_holdout


## 3. Clone or refresh the repository branch

In [3]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(PROJECT_DIR)

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Project directory:", PROJECT_DIR)
print("Working directory:", Path.cwd())

Cloning into '/content/DiverseVul--IS-Project'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 53 (delta 6), reused 42 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 638.21 KiB | 3.29 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Repository: /content/DiverseVul--IS-Project
Branch: prashant
Project directory: /content/DiverseVul--IS-Project/vuln-detection
Working directory: /content/DiverseVul--IS-Project/vuln-detection


## 4. Install runtime dependencies

In [4]:
!pip -q install \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    matplotlib \
    pyarrow \
    joblib

## 5. Import project modules and verify the expected interfaces


In [5]:
import importlib
import json
import numpy as np
import pandas as pd


exp0_lr = importlib.import_module(
    "case_study_1.exp0.exp0_lr"
)

split_manifest = importlib.import_module(
    "case_study_1.split_manifest"
)

evaluation = importlib.import_module(
    "case_study_1.evaluation"
)

dataset_loader = importlib.import_module(
    "case_study_1.dataset_loader"
)

normalization = importlib.import_module(
    "case_study_1.normalization"
)

model_selection = importlib.import_module(
    "case_study_1.model_selection"
)

required_exp0_api = [
    "EXP0_VERSION",
    "Exp0Config",
    "run_exp0_profile_fold",
    "run_exp0",
    "run_exp0_final_holdout",
]

missing_exp0_api = [
    attribute
    for attribute in required_exp0_api
    if not hasattr(exp0_lr, attribute)
]

if missing_exp0_api:
    raise AttributeError(
        "The imported EXP-0 module is missing required functionality: "
        f"{missing_exp0_api}. Ensure the latest exp0_lr.py is committed."
    )

print("✅ EXP-0 module:", exp0_lr.__file__)
print("✅ EXP-0 version:", exp0_lr.EXP0_VERSION)

print("✅ Split-manifest module:", split_manifest.__file__)
print("✅ Manifest version:", split_manifest.MANIFEST_VERSION)

print("✅ Evaluation module:", evaluation.__file__)
print("✅ Evaluation version:", evaluation.EVALUATION_VERSION)

print("✅ Dataset-loader module:", dataset_loader.__file__)
print("✅ Normalization module:", normalization.__file__)
print("✅ Model-selection module:", model_selection.__file__)

✅ EXP-0 module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp0/exp0_lr.py
✅ EXP-0 version: cs1-exp0-sgd-logistic-v2-holdout-innercv
✅ Split-manifest module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/split_manifest.py
✅ Manifest version: cs1-project-grouped-5fold-v1
✅ Evaluation module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/evaluation.py
✅ Evaluation version: cs1-evaluation-v1
✅ Dataset-loader module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/dataset_loader.py
✅ Normalization module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/normalization.py
✅ Model-selection module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/model_selection.py


## 6. Locate the raw RDiverseVul file

In [6]:
raw_file_candidates = [
    path
    for path in RAW_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".json", ".jsonl", ".ndjson", ".csv", ".parquet"}
]

if not raw_file_candidates:
    raise FileNotFoundError(
        f"No supported dataset file found inside {RAW_DIR}"
    )

preferred_names = {
    "rdiversevul.json",
    "rdiversevul.jsonl",
    "rdiversevul.parquet",
}

preferred_candidates = [
    path
    for path in raw_file_candidates
    if path.name.lower() in preferred_names
]

DATASET_PATH = (
    preferred_candidates[0]
    if preferred_candidates
    else raw_file_candidates[0]
)

print("Raw data files:")
for path in sorted(raw_file_candidates):
    print(" -", path.name)

print("\nSelected dataset:", DATASET_PATH)

Raw data files:
 - rdiversevul.json

Selected dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/raw/rdiversevul.json


## 7. Load and audit the raw dataset

In [7]:
format_audit_report = dataset_loader.format_audit_report
load_and_audit = dataset_loader.load_and_audit

dataset_df, audit_report = load_and_audit(
    path=DATASET_PATH,
    audit_output_path=AUDIT_OUTPUT_PATH,
    require_project=True,
)

print(format_audit_report(audit_report))
print("\nAudit report saved to:", AUDIT_OUTPUT_PATH)

CASE STUDY 1 — DATASET AUDIT
Rows loaded:                 261,667
Fully usable grouped-CV rows: 261,667
Vulnerable / non-vulnerable: 13,938 / 247,729
Vulnerable rate:             0.053266
Unique projects:             797
Exact duplicate excess rows: 0
Conflicting duplicate hashes: 0
Code length (characters):    median=475.0, q75=1085.0, max=240968
Warnings:
  - Positive class is rare; accuracy must not be a primary metric.

Audit report saved to: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/setup/dataset_audit.json


## 8. Build or load the conservative normalized cache

Normalization is deterministic and label-independent, so it can be performed
before the outer holdout is created. It is retained only to make lexical
features consistent; it does not fit vocabulary, IDF weights, a scaler, or a
model on the full dataset.

In [8]:
import json

EXPECTED_ROWS = 261_667

if NORMALIZED_DATA_PATH.exists():
    normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)
    print("✅ Loaded existing normalized cache.")
else:
    print("Normalized cache not found. Building it now...")

    normalized_df = normalization.add_normalized_code_column(
        frame=dataset_df,
        source_column="code",
        target_column="normalized_code",
    )

    normalization_report = normalization.normalization_summary(
        raw_codes=normalized_df["code"],
        normalized_codes=normalized_df["normalized_code"],
    )

    normalized_df = normalized_df[
        [
            "source_row_id",
            "code",
            "normalized_code",
            "label",
            "project",
        ]
    ].copy()

    normalized_df.to_parquet(NORMALIZED_DATA_PATH, index=False)

    with NORMALIZATION_REPORT_PATH.open("w", encoding="utf-8") as file:
        json.dump(normalization_report, file, indent=2)

    print("✅ Saved normalized cache:", NORMALIZED_DATA_PATH)
    print("✅ Saved normalization summary:", NORMALIZATION_REPORT_PATH)

required_columns = {
    "source_row_id",
    "code",
    "normalized_code",
    "label",
    "project",
}

missing_columns = required_columns - set(normalized_df.columns)
if missing_columns:
    raise ValueError(
        f"Normalized cache is missing required columns: {sorted(missing_columns)}"
    )

assert normalized_df["source_row_id"].nunique() == len(normalized_df)
assert normalized_df["normalized_code"].notna().all()
assert (normalized_df["normalized_code"].str.strip() != "").all()
assert set(normalized_df["label"].astype(int).unique()) == {0, 1}
assert normalized_df["project"].notna().all()
assert normalized_df["project"].astype(str).str.strip().ne("").all()

if len(normalized_df) != EXPECTED_ROWS:
    print(
        f"⚠️ Expected {EXPECTED_ROWS:,} rows from the current RDiverseVul setup, "
        f"but found {len(normalized_df):,}. The notebook will continue only if "
        "this is intentional."
    )
else:
    print(f"✅ Row count matches expected dataset size: {EXPECTED_ROWS:,}")

print("Normalized data shape:", normalized_df.shape)
print("Projects:", normalized_df["project"].nunique())
print("Vulnerable rate:", f"{normalized_df['label'].mean():.4%}")

✅ Loaded existing normalized cache.
✅ Row count matches expected dataset size: 261,667
Normalized data shape: (261667, 5)
Projects: 797
Vulnerable rate: 5.3266%


## 9. Create or load the outer project-disjoint holdout

The outer holdout must be fixed before model development. Whole projects are
kept together, so its size is approximate rather than exactly 20% of rows.

To avoid arbitrarily choosing the first grouped split, the notebook evaluates
all five `StratifiedGroupKFold` candidate test folds and deterministically
chooses the candidate closest to:

```text
target row share = 20%
target prevalence = full-dataset prevalence
```

The selection uses only split-balance statistics, never model performance.

In [9]:
from datetime import datetime, timezone
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

OUTER_N_SPLITS = 5
OUTER_RANDOM_STATE = 2026
OUTER_TARGET_ROW_SHARE = 1.0 / OUTER_N_SPLITS

def validate_outer_manifest(manifest, reference_frame):
    required_outer_columns = {
        "source_row_id",
        "label",
        "project",
        "partition",
        "outer_holdout_fold",
    }

    missing = required_outer_columns - set(manifest.columns)
    if missing:
        raise ValueError(
            f"Outer manifest is missing columns: {sorted(missing)}"
        )

    if manifest["source_row_id"].duplicated().any():
        raise ValueError("Outer manifest contains duplicate source_row_id values.")

    if set(manifest["source_row_id"]) != set(reference_frame["source_row_id"]):
        raise ValueError(
            "Outer manifest source_row_id values do not exactly match normalized data."
        )

    if set(manifest["partition"].unique()) != {"development", "outer_holdout"}:
        raise ValueError(
            "Outer manifest must contain exactly development and outer_holdout partitions."
        )

    holdout = manifest.loc[
        manifest["partition"] == "outer_holdout"
    ]
    development = manifest.loc[
        manifest["partition"] == "development"
    ]

    if holdout.empty or development.empty:
        raise ValueError("Outer holdout or development partition is empty.")

    if set(holdout["label"].astype(int).unique()) != {0, 1}:
        raise ValueError("Outer holdout does not contain both classes.")

    if set(development["label"].astype(int).unique()) != {0, 1}:
        raise ValueError("Development partition does not contain both classes.")

    holdout_projects = set(holdout["project"].astype(str))
    development_projects = set(development["project"].astype(str))

    overlap = holdout_projects.intersection(development_projects)

    if overlap:
        raise ValueError(
            "Project leakage between development and outer holdout. "
            f"Examples: {sorted(overlap)[:10]}"
        )

    return holdout, development

if OUTER_MANIFEST_PATH.exists():
    outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
    outer_holdout_frame, outer_development_frame = validate_outer_manifest(
        outer_manifest_df,
        normalized_df,
    )

    print("✅ Loaded existing frozen outer holdout manifest.")
else:
    splitter = StratifiedGroupKFold(
        n_splits=OUTER_N_SPLITS,
        shuffle=True,
        random_state=OUTER_RANDOM_STATE,
    )

    y = normalized_df["label"].to_numpy(dtype=np.int8)
    groups = normalized_df["project"].astype(str).to_numpy()
    dummy_x = np.zeros((len(normalized_df), 1), dtype=np.uint8)

    global_positive_rate = float(y.mean())
    candidate_positions = {}
    candidate_rows = []

    for candidate_fold_id, (_, test_positions) in enumerate(
        splitter.split(dummy_x, y, groups)
    ):
        candidate_positions[candidate_fold_id] = test_positions

        candidate_labels = y[test_positions]
        holdout_share = float(len(test_positions) / len(normalized_df))
        holdout_positive_rate = float(candidate_labels.mean())

        # Dimensionless score. Lower is better.
        row_share_error = abs(holdout_share - OUTER_TARGET_ROW_SHARE) / OUTER_TARGET_ROW_SHARE
        prevalence_error = abs(holdout_positive_rate - global_positive_rate) / max(
            global_positive_rate,
            1e-12,
        )

        candidate_rows.append(
            {
                "candidate_outer_fold": int(candidate_fold_id),
                "holdout_rows": int(len(test_positions)),
                "holdout_row_share": holdout_share,
                "holdout_vulnerable": int(candidate_labels.sum()),
                "holdout_non_vulnerable": int(len(candidate_labels) - candidate_labels.sum()),
                "holdout_positive_rate": holdout_positive_rate,
                "holdout_unique_projects": int(
                    normalized_df.iloc[test_positions]["project"].nunique()
                ),
                "row_share_relative_error": row_share_error,
                "positive_rate_relative_error": prevalence_error,
                "balance_score": row_share_error + prevalence_error,
            }
        )

    outer_candidates_df = (
        pd.DataFrame(candidate_rows)
        .sort_values(
            [
                "balance_score",
                "row_share_relative_error",
                "positive_rate_relative_error",
                "candidate_outer_fold",
            ]
        )
        .reset_index(drop=True)
    )

    selected_outer_fold = int(
        outer_candidates_df.iloc[0]["candidate_outer_fold"]
    )

    selected_test_positions = candidate_positions[selected_outer_fold]

    outer_manifest_df = normalized_df[
        [
            "source_row_id",
            "label",
            "project",
        ]
    ].copy()

    outer_manifest_df["partition"] = "development"
    outer_manifest_df["outer_holdout_fold"] = -1

    selected_row_indices = normalized_df.index[selected_test_positions]

    outer_manifest_df.loc[
        selected_row_indices,
        "partition",
    ] = "outer_holdout"

    outer_manifest_df.loc[
        selected_row_indices,
        "outer_holdout_fold",
    ] = selected_outer_fold

    outer_holdout_frame, outer_development_frame = validate_outer_manifest(
        outer_manifest_df,
        normalized_df,
    )

    outer_manifest_df = (
        outer_manifest_df
        .sort_values("source_row_id", kind="stable")
        .reset_index(drop=True)
    )

    outer_candidates_df.to_csv(OUTER_CANDIDATES_PATH, index=False)
    outer_manifest_df.to_parquet(OUTER_MANIFEST_PATH, index=False)
    outer_manifest_df.to_csv(
        OUTER_MANIFEST_PATH.with_suffix(".csv"),
        index=False,
    )

    metadata = {
        "outer_splitter": "StratifiedGroupKFold",
        "outer_n_splits": OUTER_N_SPLITS,
        "outer_random_state": OUTER_RANDOM_STATE,
        "target_row_share": OUTER_TARGET_ROW_SHARE,
        "selection_rule": (
            "Choose the candidate fold minimizing row-share relative error "
            "+ positive-rate relative error. No model performance was used."
        ),
        "selected_outer_holdout_fold": selected_outer_fold,
        "rows": int(len(outer_manifest_df)),
        "outer_holdout_rows": int(len(outer_holdout_frame)),
        "development_rows": int(len(outer_development_frame)),
        "outer_holdout_row_share": float(
            len(outer_holdout_frame) / len(outer_manifest_df)
        ),
        "outer_holdout_positive_rate": float(
            outer_holdout_frame["label"].mean()
        ),
        "development_positive_rate": float(
            outer_development_frame["label"].mean()
        ),
        "outer_holdout_projects": int(
            outer_holdout_frame["project"].nunique()
        ),
        "development_projects": int(
            outer_development_frame["project"].nunique()
        ),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }

    with OUTER_METADATA_PATH.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    print("✅ Created and froze the outer project holdout.")
    print("Selected candidate outer fold:", selected_outer_fold)
    print("Saved candidate audit:", OUTER_CANDIDATES_PATH)
    print("Saved outer manifest:", OUTER_MANIFEST_PATH)
    print("Saved outer metadata:", OUTER_METADATA_PATH)

# Rebuild canonical dataframes from the frozen manifest in all cases.
outer_holdout_ids = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "outer_holdout",
        "source_row_id",
    ]
)

holdout_df = normalized_df.loc[
    normalized_df["source_row_id"].isin(outer_holdout_ids)
].reset_index(drop=True)

dev_df = normalized_df.loc[
    ~normalized_df["source_row_id"].isin(outer_holdout_ids)
].reset_index(drop=True)

assert set(dev_df["source_row_id"]).isdisjoint(
    set(holdout_df["source_row_id"])
)

assert set(dev_df["source_row_id"]).union(
    set(holdout_df["source_row_id"])
) == set(normalized_df["source_row_id"])

assert set(dev_df["project"]).isdisjoint(
    set(holdout_df["project"])
)

print("\nOuter split summary")
print("=" * 72)
print(f"Full rows:              {len(normalized_df):,}")
print(f"Development rows:       {len(dev_df):,} ({len(dev_df) / len(normalized_df):.2%})")
print(f"Outer holdout rows:     {len(holdout_df):,} ({len(holdout_df) / len(normalized_df):.2%})")
print(f"Development projects:   {dev_df['project'].nunique():,}")
print(f"Outer holdout projects: {holdout_df['project'].nunique():,}")
print(f"Development prevalence: {dev_df['label'].mean():.4%}")
print(f"Holdout prevalence:     {holdout_df['label'].mean():.4%}")
print("✅ Development and outer holdout are project-disjoint.")

✅ Loaded existing frozen outer holdout manifest.

Outer split summary
Full rows:              261,667
Development rows:       203,958 (77.95%)
Outer holdout rows:     57,709 (22.05%)
Development projects:   594
Outer holdout projects: 203
Development prevalence: 5.2594%
Holdout prevalence:     5.5641%
✅ Development and outer holdout are project-disjoint.


## 10. Inspect the candidate outer splits and final partition

In [10]:
from IPython.display import display

if OUTER_CANDIDATES_PATH.exists():
    outer_candidates_df = pd.read_csv(OUTER_CANDIDATES_PATH)
    display(
        outer_candidates_df.style.format(
            {
                "holdout_row_share": "{:.2%}",
                "holdout_positive_rate": "{:.4%}",
                "row_share_relative_error": "{:.4f}",
                "positive_rate_relative_error": "{:.4f}",
                "balance_score": "{:.4f}",
            }
        )
    )
else:
    print(
        "Candidate audit file is unavailable because this notebook loaded "
        "an older frozen outer manifest. The manifest itself remains valid."
    )

partition_summary = pd.DataFrame(
    [
        {
            "partition": "development",
            "rows": len(dev_df),
            "row_share": len(dev_df) / len(normalized_df),
            "projects": dev_df["project"].nunique(),
            "vulnerable": int((dev_df["label"] == 1).sum()),
            "positive_rate": dev_df["label"].mean(),
        },
        {
            "partition": "outer_holdout",
            "rows": len(holdout_df),
            "row_share": len(holdout_df) / len(normalized_df),
            "projects": holdout_df["project"].nunique(),
            "vulnerable": int((holdout_df["label"] == 1).sum()),
            "positive_rate": holdout_df["label"].mean(),
        },
    ]
)

display(
    partition_summary.style.format(
        {
            "row_share": "{:.2%}",
            "positive_rate": "{:.4%}",
        }
    )
)

,candidate_outer_fold,holdout_rows,holdout_row_share,holdout_vulnerable,holdout_non_vulnerable,holdout_positive_rate,holdout_unique_projects,row_share_relative_error,positive_rate_relative_error,balance_score
0,2,57709,22.05%,3211,54498,5.5641%,203,0.1027,0.0446,0.1473
1,4,55428,21.18%,3305,52123,5.9627%,180,0.0591,0.1194,0.1785
2,1,38030,14.53%,2522,35508,6.6316%,191,0.2733,0.2450,0.5183
3,0,7472,2.86%,428,7044,5.7281%,48,0.8572,0.0754,0.9326
4,3,103028,39.37%,4472,98556,4.3406%,175,0.9687,0.1851,1.1538


,partition,rows,row_share,projects,vulnerable,positive_rate
0,development,203958,77.95%,594,10727,5.2594%
1,outer_holdout,57709,22.05%,203,3211,5.5641%


## 11. Audit exact normalized-code duplicates across the outer split

Project-disjoint splitting prevents project leakage. This additional audit checks
whether *identical normalized source code* occurs in both development and
outer-holdout partitions.

The notebook reports the count but does not silently delete data or change labels.
If cross-partition duplicates are material, record this limitation and consider a
component-aware duplicate-group split in a later hardening step.

In [11]:
# 11. Audit exact normalized-code duplicates across the outer split

from hashlib import sha256
import pandas as pd

# Build a compact table for duplicate analysis.
duplicate_audit_df = normalized_df[
    [
        "source_row_id",
        "label",
        "normalized_code",
        "project",
    ]
].copy()

# Hash normalized source code so exact duplicate groups can be compared
# without repeatedly storing or grouping very long C/C++ strings.
duplicate_audit_df["code_hash"] = duplicate_audit_df[
    "normalized_code"
].map(
    lambda text: sha256(
        str(text).encode("utf-8", errors="replace")
    ).hexdigest()
)

# Attach the frozen outer-split partition.
# IMPORTANT:
# - how="left" means keep every normalized dataset row.
# - validate="one_to_one" verifies that each source_row_id occurs exactly once
#   in both dataframes.
duplicate_audit_df = duplicate_audit_df.merge(
    outer_manifest_df[
        [
            "source_row_id",
            "partition",
        ]
    ],
    on="source_row_id",
    how="left",
    validate="one_to_one",
)

# Every source row must belong to exactly one outer partition.
if duplicate_audit_df["partition"].isna().any():
    missing_partition_rows = int(
        duplicate_audit_df["partition"].isna().sum()
    )
    raise ValueError(
        f"{missing_partition_rows:,} rows have no partition assignment in "
        "outer_manifest_df."
    )

if set(duplicate_audit_df["partition"].unique()) != {
    "development",
    "outer_holdout",
}:
    raise ValueError(
        "Expected exactly two partitions: development and outer_holdout. "
        f"Observed: {sorted(duplicate_audit_df['partition'].unique())}"
    )

# A code hash crosses the outer split when the same normalized source code
# appears in both development and outer-holdout partitions.
hash_partition_counts = (
    duplicate_audit_df
    .groupby("code_hash")["partition"]
    .nunique()
)

cross_partition_hashes = set(
    hash_partition_counts[
        hash_partition_counts > 1
    ].index
)

cross_partition_duplicate_rows = (
    duplicate_audit_df.loc[
        duplicate_audit_df["code_hash"].isin(cross_partition_hashes)
    ]
    .sort_values(
        [
            "code_hash",
            "partition",
            "source_row_id",
        ]
    )
    .reset_index(drop=True)
)

# Group-level summary: one row per duplicate-code hash.
cross_partition_duplicate_groups = (
    cross_partition_duplicate_rows
    .groupby("code_hash", as_index=False)
    .agg(
        total_rows=("source_row_id", "size"),
        unique_projects=("project", "nunique"),
        partitions=("partition", lambda values: ", ".join(sorted(set(values)))),
        labels_present=("label", lambda values: ", ".join(
            map(str, sorted(set(values.astype(int))))
        )),
        development_rows=(
            "partition",
            lambda values: int((values == "development").sum()),
        ),
        holdout_rows=(
            "partition",
            lambda values: int((values == "outer_holdout").sum()),
        ),
    )
    .sort_values(
        [
            "total_rows",
            "unique_projects",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

DUPLICATE_ROWS_AUDIT_PATH = (
    OUTER_SPLIT_DIR
    / "cs1_outer_cross_partition_duplicate_rows.csv"
)

DUPLICATE_GROUPS_AUDIT_PATH = (
    OUTER_SPLIT_DIR
    / "cs1_outer_cross_partition_duplicate_groups.csv"
)

cross_partition_duplicate_rows.to_csv(
    DUPLICATE_ROWS_AUDIT_PATH,
    index=False,
)

cross_partition_duplicate_groups.to_csv(
    DUPLICATE_GROUPS_AUDIT_PATH,
    index=False,
)

print("=" * 72)
print("OUTER-SPLIT EXACT NORMALIZED-CODE DUPLICATE AUDIT")
print("=" * 72)
print(
    "Exact normalized-code hashes crossing dev/holdout:",
    f"{len(cross_partition_hashes):,}",
)
print(
    "Rows involved in cross-partition duplicate groups:",
    f"{len(cross_partition_duplicate_rows):,}",
)
print(
    "Duplicate groups saved to:",
    DUPLICATE_GROUPS_AUDIT_PATH,
)
print(
    "Duplicate rows saved to:",
    DUPLICATE_ROWS_AUDIT_PATH,
)

if len(cross_partition_hashes) == 0:
    print("\n✅ No exact normalized-code duplicate crosses development and outer holdout.")
else:
    print(
        "\n⚠️ Exact normalized-code duplicates cross development and outer holdout."
    )
    print(
        "This does not invalidate the project-disjoint split, but it can allow "
        "near-direct lexical leakage across partitions. Keep the audit files "
        "and decide later whether a duplicate-component-aware split is needed."
    )

display(cross_partition_duplicate_groups.head(20))

OUTER-SPLIT EXACT NORMALIZED-CODE DUPLICATE AUDIT
Exact normalized-code hashes crossing dev/holdout: 0
Rows involved in cross-partition duplicate groups: 0
Duplicate groups saved to: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_cross_partition_duplicate_groups.csv
Duplicate rows saved to: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_cross_partition_duplicate_rows.csv

✅ No exact normalized-code duplicate crosses development and outer holdout.


,code_hash,total_rows,unique_projects,partitions,labels_present,development_rows,holdout_rows


## 12. Create or load the inner 5-fold project-grouped development manifest

The inner manifest includes **only** `dev_df`.

It is the manifest used for EXP-0 / EXP-1 / EXP-2 model selection. The outer
holdout is absent from every inner fold and must not receive any OOF score at
this stage.

In [14]:
# 12. Create or rebuild the inner 5-fold project-grouped development manifest
#
# Important:
# - This uses only dev_df.
# - It never touches holdout_df.
# - It searches deterministic grouped-CV seeds.
# - It rejects mathematically valid but practically unusable tiny folds.
# - It selects the best split using split statistics only, never model results.

from datetime import datetime, timezone
import json
import shutil
import pandas as pd
from IPython.display import display

INNER_N_SPLITS = 5

# This is only the model seed used later by SGD Logistic Regression.
# It is intentionally separate from the selected split seed.
INNER_MODEL_RANDOM_STATE = 42

# For this current run, rebuild because the existing manifest contains
# Fold 0 with only 128 rows.
FORCE_REBUILD_INNER_MANIFEST = False

# Deterministic seed search.
INNER_SPLIT_SEED_START = 42
INNER_SPLIT_SEED_COUNT = 1000

# A 5-fold split should ideally be near 20% per fold.
# These guardrails reject tiny or excessively dominant folds.
MIN_TEST_ROW_SHARE = 0.10   # at least 10% of development rows
MAX_TEST_ROW_SHARE = 0.35   # at most 35% of development rows

INNER_CANDIDATE_AUDIT_PATH = (
    INNER_SPLIT_DIR / "cs1_inner_grouped_seed_search_audit.csv"
)

INNER_SELECTION_METADATA_PATH = (
    INNER_SPLIT_DIR / "cs1_inner_grouped_split_selection_metadata.json"
)

# Check whether 5 grouped test folds can each contain positives.
positive_project_count = int(
    (
        dev_df.groupby("project")["label"]
        .max()
        .astype(int)
        == 1
    ).sum()
)

if positive_project_count < INNER_N_SPLITS:
    raise RuntimeError(
        "A 5-fold project-grouped CV with vulnerable samples in every test "
        "fold is impossible because development data contains only "
        f"{positive_project_count} vulnerable project(s). "
        f"At least {INNER_N_SPLITS} are required."
    )

def evaluate_inner_split_candidate(candidate_manifest, candidate_config):
    """
    Return fold-level balance diagnostics for one candidate grouped split.

    Raises ValueError when the split is not usable for this project.
    """
    summary = split_manifest.summarize_manifest(
        candidate_manifest,
        config=candidate_config,
    )

    if not (summary["train_test_project_overlap"] == 0).all():
        raise ValueError("Candidate has project overlap between train and test.")

    if not (summary["test_vulnerable"] > 0).all():
        raise ValueError("At least one fold has no vulnerable sample.")

    if not (summary["test_non_vulnerable"] > 0).all():
        raise ValueError("At least one fold has no non-vulnerable sample.")

    min_row_share = float(summary["test_row_share"].min())
    max_row_share = float(summary["test_row_share"].max())

    if min_row_share < MIN_TEST_ROW_SHARE:
        raise ValueError(
            f"Smallest fold row share is {min_row_share:.4%}, below the "
            f"required minimum {MIN_TEST_ROW_SHARE:.2%}."
        )

    if max_row_share > MAX_TEST_ROW_SHARE:
        raise ValueError(
            f"Largest fold row share is {max_row_share:.4%}, above the "
            f"allowed maximum {MAX_TEST_ROW_SHARE:.2%}."
        )

    global_positive_rate = float(dev_df["label"].mean())
    target_row_share = 1.0 / INNER_N_SPLITS

    max_row_share_deviation = float(
        (
            summary["test_row_share"] - target_row_share
        ).abs().max()
    )

    std_row_share = float(
        summary["test_row_share"].std(ddof=0)
    )

    max_positive_rate_deviation = float(
        (
            summary["test_positive_rate"] - global_positive_rate
        ).abs().max()
    )

    std_positive_rate = float(
        summary["test_positive_rate"].std(ddof=0)
    )

    return {
        "summary": summary,
        "min_test_rows": int(summary["test_rows"].min()),
        "max_test_rows": int(summary["test_rows"].max()),
        "min_test_row_share": min_row_share,
        "max_test_row_share": max_row_share,
        "min_test_vulnerable": int(summary["test_vulnerable"].min()),
        "max_test_vulnerable": int(summary["test_vulnerable"].max()),
        "min_test_positive_rate": float(
            summary["test_positive_rate"].min()
        ),
        "max_test_positive_rate": float(
            summary["test_positive_rate"].max()
        ),
        "max_row_share_deviation": max_row_share_deviation,
        "std_row_share": std_row_share,
        "max_positive_rate_deviation": max_positive_rate_deviation,
        "std_positive_rate": std_positive_rate,
    }

# ------------------------------------------------------------------
# Load an already-valid frozen manifest only when rebuilding is disabled.
# ------------------------------------------------------------------
if INNER_MANIFEST_PATH.exists() and not FORCE_REBUILD_INNER_MANIFEST:
    inner_manifest_df = split_manifest.load_manifest(
        INNER_MANIFEST_PATH,
        config=split_manifest.SplitConfig(
            n_splits=INNER_N_SPLITS,
            random_state=42,
            shuffle=True,
        ),
    )

    selected_inner_split_seed = None

    if INNER_SELECTION_METADATA_PATH.exists():
        with INNER_SELECTION_METADATA_PATH.open(
            "r",
            encoding="utf-8",
        ) as file:
            inner_selection_metadata = json.load(file)

        selected_inner_split_seed = inner_selection_metadata.get(
            "selected_split_seed"
        )

    selected_inner_split_config = split_manifest.SplitConfig(
        n_splits=INNER_N_SPLITS,
        random_state=(
            selected_inner_split_seed
            if selected_inner_split_seed is not None
            else 42
        ),
        shuffle=True,
    )

    selected_diagnostics = evaluate_inner_split_candidate(
        inner_manifest_df,
        selected_inner_split_config,
    )

    print("✅ Loaded existing valid frozen inner development manifest.")
    print(
        "Selected inner split seed:",
        selected_inner_split_seed
        if selected_inner_split_seed is not None
        else "not recorded",
    )

# ------------------------------------------------------------------
# Build a new valid frozen manifest.
# ------------------------------------------------------------------
else:
    print(
        "Searching deterministic StratifiedGroupKFold seeds for a "
        "balanced and valid 5-fold development split..."
    )

    # Preserve the bad manifest rather than silently losing it.
    if INNER_MANIFEST_PATH.exists():
        backup_path = INNER_MANIFEST_PATH.with_name(
            "cs1_inner_project_grouped_5fold_manifest_rejected_tiny_fold.parquet"
        )

        if not backup_path.exists():
            shutil.copy2(INNER_MANIFEST_PATH, backup_path)
            print("Archived rejected tiny-fold manifest to:", backup_path)

    candidate_rows = []
    best_manifest = None
    best_seed = None
    best_diagnostics = None
    best_key = None

    for offset in range(INNER_SPLIT_SEED_COUNT):
        candidate_seed = INNER_SPLIT_SEED_START + offset

        candidate_config = split_manifest.SplitConfig(
            n_splits=INNER_N_SPLITS,
            random_state=candidate_seed,
            shuffle=True,
        )

        try:
            candidate_manifest = (
                split_manifest.create_project_grouped_manifest(
                    dev_df,
                    config=candidate_config,
                )
            )

            diagnostics = evaluate_inner_split_candidate(
                candidate_manifest,
                candidate_config,
            )

            # Selection priority:
            # 1. row-size balance across folds
            # 2. prevalence balance across folds
            # 3. deterministic seed tie-break
            ranking_key = (
                diagnostics["max_row_share_deviation"],
                diagnostics["std_row_share"],
                diagnostics["max_positive_rate_deviation"],
                diagnostics["std_positive_rate"],
                candidate_seed,
            )

            candidate_rows.append(
                {
                    "split_seed": candidate_seed,
                    "eligible": True,
                    "failure_reason": "",
                    **{
                        key: value
                        for key, value in diagnostics.items()
                        if key != "summary"
                    },
                }
            )

            if best_key is None or ranking_key < best_key:
                best_key = ranking_key
                best_seed = candidate_seed
                best_manifest = candidate_manifest.copy()
                best_diagnostics = diagnostics

        except (ValueError, RuntimeError) as exc:
            candidate_rows.append(
                {
                    "split_seed": candidate_seed,
                    "eligible": False,
                    "failure_reason": str(exc),
                    "min_test_rows": pd.NA,
                    "max_test_rows": pd.NA,
                    "min_test_row_share": pd.NA,
                    "max_test_row_share": pd.NA,
                    "min_test_vulnerable": pd.NA,
                    "max_test_vulnerable": pd.NA,
                    "min_test_positive_rate": pd.NA,
                    "max_test_positive_rate": pd.NA,
                    "max_row_share_deviation": pd.NA,
                    "std_row_share": pd.NA,
                    "max_positive_rate_deviation": pd.NA,
                    "std_positive_rate": pd.NA,
                }
            )

    inner_candidate_audit_df = pd.DataFrame(candidate_rows)

    eligible_candidates_df = (
        inner_candidate_audit_df.loc[
            inner_candidate_audit_df["eligible"]
        ]
        .sort_values(
            [
                "max_row_share_deviation",
                "std_row_share",
                "max_positive_rate_deviation",
                "std_positive_rate",
                "split_seed",
            ]
        )
        .reset_index(drop=True)
    )

    inner_candidate_audit_df.to_csv(
        INNER_CANDIDATE_AUDIT_PATH,
        index=False,
    )

    if best_manifest is None:
        print("\nTop failed candidates:")
        display(
            inner_candidate_audit_df[
                [
                    "split_seed",
                    "failure_reason",
                ]
            ].head(20)
        )

        raise RuntimeError(
            "No acceptable 5-fold project-grouped inner split was found. "
            "The candidate audit has been saved here:\n"
            f"{INNER_CANDIDATE_AUDIT_PATH}\n\n"
            "Do not run the experiment using a tiny or one-class fold. "
            "We must then use a more advanced constrained group splitter "
            "rather than weaken project isolation."
        )

    selected_inner_split_seed = int(best_seed)

    selected_inner_split_config = split_manifest.SplitConfig(
        n_splits=INNER_N_SPLITS,
        random_state=selected_inner_split_seed,
        shuffle=True,
    )

    inner_manifest_df = (
        best_manifest
        .sort_values("source_row_id", kind="stable")
        .reset_index(drop=True)
    )

    # Final strict validation of the selected candidate.
    split_manifest.assert_manifest_integrity(
        inner_manifest_df,
        config=selected_inner_split_config,
    )

    inner_fold_summary = best_diagnostics["summary"]

    development_fingerprint = split_manifest.dataset_split_fingerprint(
        dev_df,
        config=selected_inner_split_config,
    )

    # Standard manifest artifacts inside this experiment-specific directory.
    split_manifest.save_manifest_artifacts(
        manifest=inner_manifest_df,
        output_dir=INNER_SPLIT_DIR,
        config=selected_inner_split_config,
        dataset_fingerprint=development_fingerprint,
        normalized_dataset_path=NORMALIZED_DATA_PATH,
    )

    # Canonical names used by the notebook.
    inner_manifest_df.to_parquet(
        INNER_MANIFEST_PATH,
        index=False,
    )

    inner_manifest_df.to_csv(
        INNER_MANIFEST_PATH.with_suffix(".csv"),
        index=False,
    )

    inner_fold_summary.to_csv(
        INNER_MANIFEST_SUMMARY_PATH,
        index=False,
    )

    selection_metadata = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "splitter": "StratifiedGroupKFold",
        "development_partition_only": True,
        "n_splits": INNER_N_SPLITS,
        "candidate_seed_start": INNER_SPLIT_SEED_START,
        "candidate_seed_count": INNER_SPLIT_SEED_COUNT,
        "selected_split_seed": selected_inner_split_seed,
        "eligible_candidate_count": int(len(eligible_candidates_df)),
        "hard_constraints": {
            "every_test_fold_has_both_classes": True,
            "minimum_test_row_share": MIN_TEST_ROW_SHARE,
            "maximum_test_row_share": MAX_TEST_ROW_SHARE,
            "project_overlap_between_train_and_test": False,
        },
        "selection_rule": (
            "Among eligible candidate seeds, minimize maximum fold row-share "
            "deviation from 1/n_splits, then row-share standard deviation, "
            "then maximum fold positive-rate deviation from global development "
            "prevalence, then positive-rate standard deviation, then seed."
        ),
        "development_rows": int(len(dev_df)),
        "development_projects": int(dev_df["project"].nunique()),
        "positive_projects_in_development": positive_project_count,
        "development_positive_rate": float(dev_df["label"].mean()),
        "selected_split_diagnostics": {
            key: value
            for key, value in best_diagnostics.items()
            if key != "summary"
        },
        "candidate_audit_path": str(INNER_CANDIDATE_AUDIT_PATH),
        "inner_manifest_path": str(INNER_MANIFEST_PATH),
    }

    with INNER_SELECTION_METADATA_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(selection_metadata, file, indent=2)

    selected_diagnostics = best_diagnostics

    print("✅ Created and froze a balanced valid inner grouped manifest.")
    print("Selected inner split seed:", selected_inner_split_seed)
    print(
        "Eligible seeds:",
        f"{len(eligible_candidates_df):,} / {INNER_SPLIT_SEED_COUNT:,}",
    )
    print(
        "Fold row-share range:",
        f"{best_diagnostics['min_test_row_share']:.2%}",
        "to",
        f"{best_diagnostics['max_test_row_share']:.2%}",
    )
    print(
        "Fold-size range:",
        f"{best_diagnostics['min_test_rows']:,}",
        "to",
        f"{best_diagnostics['max_test_rows']:,}",
    )
    print("Candidate audit:", INNER_CANDIDATE_AUDIT_PATH)
    print("Manifest:", INNER_MANIFEST_PATH)

# ------------------------------------------------------------------
# Common validation: applies to a loaded or newly built manifest.
# ------------------------------------------------------------------
inner_fold_summary = split_manifest.summarize_manifest(
    inner_manifest_df,
    config=selected_inner_split_config,
)

assert len(inner_manifest_df) == len(dev_df)
assert inner_manifest_df["source_row_id"].nunique() == len(dev_df)

assert set(inner_manifest_df["source_row_id"]) == set(
    dev_df["source_row_id"]
)

assert set(inner_manifest_df["source_row_id"]).isdisjoint(
    set(holdout_df["source_row_id"])
)

assert (inner_fold_summary["train_test_project_overlap"] == 0).all()
assert (inner_fold_summary["test_vulnerable"] > 0).all()
assert (inner_fold_summary["test_non_vulnerable"] > 0).all()

assert (
    inner_fold_summary["test_row_share"] >= MIN_TEST_ROW_SHARE
).all()

assert (
    inner_fold_summary["test_row_share"] <= MAX_TEST_ROW_SHARE
).all()

print("\nFinal inner development fold summary")
display(
    inner_fold_summary.style.format(
        {
            "test_positive_rate": "{:.4%}",
            "positive_rate_delta_from_global": "{:+.4%}",
            "test_row_share": "{:.2%}",
            "test_project_share": "{:.2%}",
        }
    )
)

Searching deterministic StratifiedGroupKFold seeds for a balanced and valid 5-fold development split...
✅ Created and froze a balanced valid inner grouped manifest.
Selected inner split seed: 674
Eligible seeds: 2 / 1,000
Fold row-share range: 16.46% to 27.22%
Fold-size range: 33,570 to 55,509
Candidate audit: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_inner_grouped_seed_search_audit.csv
Manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet

Final inner development fold summary


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,55509,2420,53089,4.3597%,-0.8998%,1,148449,593,0,27.22%,0.17%
1,1,40791,2091,38700,5.1261%,-0.1333%,158,163167,436,0,20.00%,26.60%
2,2,39072,2062,37010,5.2774%,+0.0180%,146,164886,448,0,19.16%,24.58%
3,3,35016,2047,32969,5.8459%,+0.5865%,137,168942,457,0,17.17%,23.06%
4,4,33570,2107,31463,6.2764%,+1.0170%,152,170388,442,0,16.46%,25.59%


## 13. Validate the inner manifest and fold coverage

In [15]:
# 13. Validate the inner manifest and fold coverage

from IPython.display import display

required_variables = [
    "dev_df",
    "holdout_df",
    "inner_manifest_df",
    "selected_inner_split_config",
    "INNER_N_SPLITS",
    "MIN_TEST_ROW_SHARE",
    "MAX_TEST_ROW_SHARE",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Section 12 must complete successfully before running Section 13. "
        f"Missing variables: {missing_variables}"
    )

# ---------------------------------------------------------------
# 1. Validate the frozen manifest itself.
# ---------------------------------------------------------------
split_manifest.assert_manifest_integrity(
    inner_manifest_df,
    config=selected_inner_split_config,
)

# ---------------------------------------------------------------
# 2. Confirm exact development-partition coverage.
# ---------------------------------------------------------------
dev_ids = set(dev_df["source_row_id"])
holdout_ids = set(holdout_df["source_row_id"])
manifest_ids = set(inner_manifest_df["source_row_id"])

assert len(inner_manifest_df) == len(dev_df), (
    "Inner manifest row count must equal development-partition row count. "
    f"Manifest={len(inner_manifest_df):,}, development={len(dev_df):,}"
)

assert inner_manifest_df["source_row_id"].nunique() == len(dev_df), (
    "Inner manifest contains duplicate source_row_id values."
)

assert manifest_ids == dev_ids, (
    "Inner manifest must contain every development sample exactly once "
    "and no extra sample."
)

assert manifest_ids.isdisjoint(holdout_ids), (
    "Outer-holdout rows were found inside the inner-CV manifest."
)

# ---------------------------------------------------------------
# 3. Confirm fold IDs and complete fold coverage.
# ---------------------------------------------------------------
observed_folds = set(
    inner_manifest_df["fold"].astype(int).unique()
)

expected_folds = set(range(INNER_N_SPLITS))

assert observed_folds == expected_folds, (
    f"Expected fold IDs {sorted(expected_folds)}, "
    f"but observed {sorted(observed_folds)}."
)

fold_counts = (
    inner_manifest_df["fold"]
    .value_counts()
    .sort_index()
)

assert len(fold_counts) == INNER_N_SPLITS
assert (fold_counts > 0).all(), (
    "At least one inner-CV fold has zero rows."
)

# ---------------------------------------------------------------
# 4. Recompute fold diagnostics from the frozen manifest.
# ---------------------------------------------------------------
inner_fold_summary = split_manifest.summarize_manifest(
    inner_manifest_df,
    config=selected_inner_split_config,
)

# Project isolation is mandatory.
assert (
    inner_fold_summary["train_test_project_overlap"] == 0
).all(), (
    "Project leakage exists between inner training and test folds."
)

# Every held-out fold must support all binary metrics.
assert (
    inner_fold_summary["test_vulnerable"] > 0
).all(), (
    "At least one inner test fold has no vulnerable samples."
)

assert (
    inner_fold_summary["test_non_vulnerable"] > 0
).all(), (
    "At least one inner test fold has no non-vulnerable samples."
)

# Practical fold-size constraints added after the rejected 128-row fold.
assert (
    inner_fold_summary["test_row_share"] >= MIN_TEST_ROW_SHARE
).all(), (
    "At least one fold is too small for meaningful grouped CV. "
    f"Minimum allowed share: {MIN_TEST_ROW_SHARE:.2%}"
)

assert (
    inner_fold_summary["test_row_share"] <= MAX_TEST_ROW_SHARE
).all(), (
    "At least one fold is excessively large for the declared grouped-CV protocol. "
    f"Maximum allowed share: {MAX_TEST_ROW_SHARE:.2%}"
)

# ---------------------------------------------------------------
# 5. Print an audit summary.
# ---------------------------------------------------------------
print("=" * 78)
print("INNER DEVELOPMENT-CV MANIFEST VALIDATION PASSED")
print("=" * 78)

print(f"Development rows:              {len(dev_df):,}")
print(f"Outer-holdout rows:            {len(holdout_df):,}")
print(f"Development projects:          {dev_df['project'].nunique():,}")
print(f"Outer-holdout projects:        {holdout_df['project'].nunique():,}")
print(f"Selected inner split seed:     {selected_inner_split_config.random_state}")
print(f"Number of inner CV folds:      {INNER_N_SPLITS}")
print(f"Minimum allowed fold share:    {MIN_TEST_ROW_SHARE:.2%}")
print(f"Maximum allowed fold share:    {MAX_TEST_ROW_SHARE:.2%}")
print("Project overlap in inner CV:   0 in every fold")
print("Outer holdout in inner CV:     0 rows")
print("Both classes in every fold:    yes")

display(
    inner_fold_summary[
        [
            "fold",
            "test_rows",
            "test_vulnerable",
            "test_non_vulnerable",
            "test_positive_rate",
            "test_unique_projects",
            "train_rows",
            "train_unique_projects",
            "train_test_project_overlap",
            "test_row_share",
            "test_project_share",
        ]
    ].style.format(
        {
            "test_positive_rate": "{:.4%}",
            "test_row_share": "{:.2%}",
            "test_project_share": "{:.2%}",
        }
    )
)

INNER DEVELOPMENT-CV MANIFEST VALIDATION PASSED
Development rows:              203,958
Outer-holdout rows:            57,709
Development projects:          594
Outer-holdout projects:        203
Selected inner split seed:     674
Number of inner CV folds:      5
Minimum allowed fold share:    10.00%
Maximum allowed fold share:    35.00%
Project overlap in inner CV:   0 in every fold
Outer holdout in inner CV:     0 rows
Both classes in every fold:    yes


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,55509,2420,53089,4.3597%,1,148449,593,0,27.22%,0.17%
1,1,40791,2091,38700,5.1261%,158,163167,436,0,20.00%,26.60%
2,2,39072,2062,37010,5.2774%,146,164886,448,0,19.16%,24.58%
3,3,35016,2047,32969,5.8459%,137,168942,457,0,17.17%,23.06%
4,4,33570,2107,31463,6.2764%,152,170388,442,0,16.46%,25.59%


## 14. Attach inner-fold assignments to development data

In [16]:
dev_with_folds_df = split_manifest.apply_manifest(
    frame=dev_df,
    manifest=inner_manifest_df,
)

assert len(dev_with_folds_df) == len(dev_df)
assert set(dev_with_folds_df["fold"].unique()) == set(
    range(INNER_N_SPLITS)
)

print("Development data with inner-fold assignment:", dev_with_folds_df.shape)

display(
    dev_with_folds_df[
        [
            "source_row_id",
            "label",
            "project",
            "fold",
            "normalized_code",
        ]
    ].head(10)
)

Development data with inner-fold assignment: (203958, 6)


,source_row_id,label,project,fold,normalized_code
0,0,0,linux,0,"void dcn20_calculate_wm(\nstruct dc *dc, struc..."
1,1,0,net,1,int hw_atl_utils_soft_reset(struct aq_hw_s *se...
2,2,0,net,1,static u32 hw_atl_utils_rpc_state_get(struct a...
3,3,0,net,1,int hw_atl_utils_hw_get_regs(struct aq_hw_s *s...
4,4,0,net,1,"int hw_atl_utils_initfw(struct aq_hw_s *self, ..."
5,5,0,net,1,static u32 aq_fw1x_rpc_get(struct aq_hw_s *sel...
6,6,0,net,1,u32 hw_atl_utils_get_fw_version(struct aq_hw_s...
7,7,0,net,1,int hw_atl_write_fwcfg_dwords(struct aq_hw_s *...
8,8,0,net,1,int hw_atl_utils_mpi_get_link_status(struct aq...
9,9,0,net,1,static int hw_atl_utils_soft_reset_flb(struct ...


## 15. Declare the frozen EXP-0 configuration

This is the baseline configuration. Do not tune it using outer-holdout results.

```text
normalized C/C++ function
→ word TF-IDF (1–3 grams)
+ character TF-IDF (3–4 grams)
→ 110,000 sparse features
→ L2 class-weighted SGD Logistic Regression
```

In [17]:
# 15. Declare the frozen EXP-0 configuration

# This random state belongs to the SGD Logistic Regression optimizer only.
# It is intentionally separate from the selected grouped-CV split seed.
#
# The split seed is already frozen inside inner_manifest_df and does not need
# to be passed to the model configuration.

INNER_MODEL_RANDOM_STATE = 42

exp0_config = exp0_lr.Exp0Config(
    experiment_name="cs1_exp0_lr_inner_dev_grouped",

    n_splits=INNER_N_SPLITS,
    random_state=INNER_MODEL_RANDOM_STATE,
    decision_threshold=0.50,

    # Word TF-IDF: identifiers, APIs, and local lexical sequences.
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,

    # Character TF-IDF: syntax fragments, pointer patterns, operators, APIs.
    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,

    # Class-weighted L2 Logistic Regression trained through SGD.
    sgd_loss="log_loss",
    sgd_penalty="l2",
    sgd_alpha=1e-5,
    sgd_class_weight="balanced",
    sgd_max_iter=80,
    sgd_tol=1e-3,
    sgd_average=True,

    top_features_per_direction=30,
    verbose=True,
)

print("EXP-0 configuration:")
print(exp0_config)

print("\nSplit protocol:")
print(f"  Inner folds: {INNER_N_SPLITS}")
print(
    "  Selected grouped-CV split seed:",
    selected_inner_split_seed
    if "selected_inner_split_seed" in globals()
    else "loaded from frozen manifest / metadata",
)
print(f"  SGD model random_state: {INNER_MODEL_RANDOM_STATE}")
print("  Outer holdout: frozen and still untouched")

EXP-0 configuration:
Exp0Config(experiment_name='cs1_exp0_lr_inner_dev_grouped', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', n_splits=5, random_state=42, decision_threshold=0.5, word_ngram_range=(1, 3), word_min_df=3, word_max_df=0.995, word_max_features=50000, char_analyzer='char', char_ngram_range=(3, 4), char_min_df=8, char_max_df=0.995, char_max_features=60000, lowercase=False, sublinear_tf=True, tfidf_norm='l2', sgd_loss='log_loss', sgd_penalty='l2', sgd_alpha=1e-05, sgd_class_weight='balanced', sgd_max_iter=80, sgd_tol=0.001, sgd_average=True, top_features_per_direction=30, verbose=True)

Split protocol:
  Inner folds: 5
  Selected grouped-CV split seed: 674
  SGD model random_state: 42
  Outer holdout: frozen and still untouched


## 16. Profile one inner-CV fold

This profile uses development data only. It verifies runtime, convergence, and
the inner split before the official five-fold development-CV run.

The metrics here are descriptive only; do not use them as final results.

In [18]:
exp0_profile = exp0_lr.run_exp0_profile_fold(
    normalized_frame=dev_df,
    manifest=inner_manifest_df,
    fold_id=0,
    config=exp0_config,
)

print("\nInner Fold 0 computational profile:")
display(
    exp0_profile["training_metadata"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_unique_projects",
            "test_unique_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "sparse_join_seconds",
            "model_fit_seconds",
            "prediction_seconds",
            "total_fold_seconds",
            "word_features",
            "char_features",
            "total_features",
            "model_n_iter",
            "convergence_warning_count",
            "optimizer",
        ]
    ]
)

print("\nInner Fold 0 descriptive metrics — not the final holdout result:")
print(
    evaluation.format_metric_report(
        exp0_profile["profile_metrics"]
    )
)

[07:16:32] CS1-EXP0 profiling mode: running Fold 1/5 only.
[07:16:33] Fold 1/5 started | train=148,449, test=55,509, train projects=593, test projects=1.
[07:16:33] Fold 1/5 | fitting word TF-IDF...
[07:18:32] Fold 1/5 | word TF-IDF done in 1.99 min (50,000 features).
[07:18:32] Fold 1/5 | fitting character TF-IDF...
[07:22:32] Fold 1/5 | character TF-IDF done in 4.00 min (60,000 features).
[07:22:32] Fold 1/5 | joining sparse feature matrices...
[07:22:33] Fold 1/5 | sparse matrices ready in 0.6s (total features: 110,000).
[07:22:34] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[07:22:43] Fold 1/5 | SGD Logistic Regression done in 9.8s (epochs=18, convergence warnings=0).
[07:22:43] Fold 1/5 | scoring held-out projects...
[07:22:43] Fold 1/5 | scoring done in 0.1s.
[07:22:44] Fold 1/5 completed in 6.17 min.
[07:22:44] Profiling run complete. These metrics are descriptive for one fold only; do not treat them as the official five-fold EXP-0 re

,fold,train_rows,test_rows,train_unique_projects,test_unique_projects,word_tfidf_seconds,char_tfidf_seconds,sparse_join_seconds,model_fit_seconds,prediction_seconds,total_fold_seconds,word_features,char_features,total_features,model_n_iter,convergence_warning_count,optimizer
0,0,148449,55509,593,1,119.197159,239.795396,0.564881,9.75736,0.062453,370.492393,50000,60000,110000,18,0,SGDClassifier(loss=log_loss)



Inner Fold 0 descriptive metrics — not the final holdout result:
Pooled Out-of-Fold Evaluation
                   n_samples: 55509
                vulnerable_1: 2420
            non_vulnerable_0: 53089
               positive_rate: 0.043597
                   threshold: 0.500000
    average_precision_pr_auc: 0.088308
                   precision: 0.107271
                      recall: 0.271901
                          f1: 0.153846
                         mcc: 0.109910
                 specificity: 0.896852
         false_positive_rate: 0.103148
               true_negative: 47613
              false_positive: 5476
              false_negative: 1762
               true_positive: 658


In [19]:
# Profile the most computationally demanding inner fold:
# the fold with the largest training partition.

PROFILE_FOLD_ID = int(
    inner_fold_summary.loc[
        inner_fold_summary["train_rows"].idxmax(),
        "fold",
    ]
)

print(
    "Profiling the largest training partition: "
    f"Fold {PROFILE_FOLD_ID} "
    f"with {int(inner_fold_summary.loc[
        inner_fold_summary['fold'] == PROFILE_FOLD_ID,
        'train_rows'
    ].iloc[0]):,} training rows."
)

profile_results = exp0_lr.run_exp0_profile_fold(
    normalized_frame=dev_df,
    manifest=inner_manifest_df,
    fold_id=PROFILE_FOLD_ID,
    config=exp0_config,
)

Profiling the largest training partition: Fold 4 with 170,388 training rows.
[07:28:55] CS1-EXP0 profiling mode: running Fold 5/5 only.
[07:28:56] Fold 5/5 started | train=170,388, test=33,570, train projects=442, test projects=152.
[07:28:56] Fold 5/5 | fitting word TF-IDF...
[07:31:14] Fold 5/5 | word TF-IDF done in 2.29 min (50,000 features).
[07:31:14] Fold 5/5 | fitting character TF-IDF...
[07:35:10] Fold 5/5 | character TF-IDF done in 3.94 min (60,000 features).
[07:35:10] Fold 5/5 | joining sparse feature matrices...
[07:35:11] Fold 5/5 | sparse matrices ready in 0.6s (total features: 110,000).
[07:35:11] Fold 5/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[07:35:23] Fold 5/5 | SGD Logistic Regression done in 12.3s (epochs=20, convergence warnings=0).
[07:35:23] Fold 5/5 | scoring held-out projects...
[07:35:23] Fold 5/5 | scoring done in 0.0s.
[07:35:24] Fold 5/5 completed in 6.46 min.
[07:35:24] Profiling run complete. These metrics are des

In [20]:
# Show the computational profile metadata.

print("\nLargest-fold computational profile:")
display(
    profile_results["training_metadata"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_unique_projects",
            "test_unique_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "sparse_join_seconds",
            "model_fit_seconds",
            "prediction_seconds",
            "total_fold_seconds",
            "word_features",
            "char_features",
            "total_features",
            "model_n_iter",
            "convergence_warning_count",
            "optimizer",
        ]
    ]
)

# Show descriptive metrics for this one held-out fold.
# These are only a feasibility/profile result, not the official five-fold result.

profile_metrics_df = (
    pd.DataFrame(
        list(profile_results["profile_metrics"].items()),
        columns=["metric", "value"],
    )
)

print("\nLargest-fold descriptive metrics — not the official inner-CV result:")
display(profile_metrics_df)

# Optional: inspect strongest model signals from this fold.
print("\nTop positive and negative TF-IDF features from this fold:")
display(
    profile_results["top_features"]
    .sort_values(["direction", "rank"])
    .head(20)
)


Largest-fold computational profile:


,fold,train_rows,test_rows,train_unique_projects,test_unique_projects,word_tfidf_seconds,char_tfidf_seconds,sparse_join_seconds,model_fit_seconds,prediction_seconds,total_fold_seconds,word_features,char_features,total_features,model_n_iter,convergence_warning_count,optimizer
0,4,170388,33570,442,152,137.407819,236.312354,0.563412,12.275733,0.042773,387.468646,50000,60000,110000,20,0,SGDClassifier(loss=log_loss)



Largest-fold descriptive metrics — not the official inner-CV result:


,metric,value
0,n_samples,33570.000000
1,vulnerable_1,2107.000000
2,non_vulnerable_0,31463.000000
3,positive_rate,0.062764
4,threshold,0.500000
5,average_precision_pr_auc,0.149860
6,precision,0.144792
7,recall,0.441386
8,f1,0.218054
9,mcc,0.164508



Top positive and negative TF-IDF features from this fold:


,fold,direction,rank,feature_type,feature,coefficient,abs_coefficient
30,4,non_vulnerable_associated,1,word,word::for i 0,-2.388958,2.388958
31,4,non_vulnerable_associated,2,word,word::void int,-2.374283,2.374283
32,4,non_vulnerable_associated,3,word,word::ap,-2.315584,2.315584
33,4,non_vulnerable_associated,4,word,word::ref,-2.050692,2.050692
34,4,non_vulnerable_associated,5,word,word::ElectronBrowserClient,-1.988548,1.988548
35,4,non_vulnerable_associated,6,char,char::e;\ni,-1.985256,1.985256
36,4,non_vulnerable_associated,7,char,char::d_p,-1.949691,1.949691
37,4,non_vulnerable_associated,8,word,word::plugin,-1.936569,1.936569
38,4,non_vulnerable_associated,9,word,word::u64,-1.920498,1.920498
39,4,non_vulnerable_associated,10,word,word::column,-1.914033,1.914033


## 17. Run the official inner 5-fold development-CV experiment

Set `RUN_INNER_CV = True` only after the profile succeeds. This run:

- uses **only development projects**;
- produces pooled OOF predictions only for the development partition;
- is used to compare/select candidate models;
- does not evaluate the outer holdout.

In [21]:
RUN_INNER_CV = True

if RUN_INNER_CV:
    existing_files = list(INNER_CV_OUTPUT_DIR.iterdir())

    if existing_files:
        raise RuntimeError(
            "The inner-CV output directory is not empty. "
            "Use a new experiment version, or inspect/remove only incomplete "
            "artifacts deliberately before rerunning."
        )

    exp0_inner_results = exp0_lr.run_exp0(
        normalized_frame=dev_df,
        manifest=inner_manifest_df,
        config=exp0_config,
        output_dir=INNER_CV_OUTPUT_DIR,
        additional_metadata={
            "evaluation_stage": "inner_development_cv",
            "evaluation_protocol": (
                "Outer project-disjoint holdout; inner 5-fold "
                "StratifiedGroupKFold on development projects."
            ),
            "outer_holdout_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
            "selection_rule": (
                "Use pooled development OOF PR-AUC for candidate comparison. "
                "Outer-holdout labels are excluded from model selection."
            ),
            "classifier": (
                "SGDClassifier(loss='log_loss'): L2-regularized, "
                "class-weighted linear Logistic Regression."
            ),
        },
    )

    print(
        evaluation.format_metric_report(
            exp0_inner_results["evaluation"]["pooled_metrics"]
        )
    )
else:
    print(
        "Inner CV is locked. Set RUN_INNER_CV = True after the profile is accepted."
    )

[07:40:30] CS1-EXP0 official run started: 5-fold grouped CV.
[07:40:30] Configuration: word<= 50,000, char<= 60,000, optimizer=SGDClassifier(log_loss), max_iter=80, alpha=1e-05.
[07:40:32] Fold 1/5 started | train=148,449, test=55,509, train projects=593, test projects=1.
[07:40:32] Fold 1/5 | fitting word TF-IDF...
[07:42:45] Fold 1/5 | word TF-IDF done in 2.22 min (50,000 features).
[07:42:45] Fold 1/5 | fitting character TF-IDF...
[07:46:35] Fold 1/5 | character TF-IDF done in 3.83 min (60,000 features).
[07:46:35] Fold 1/5 | joining sparse feature matrices...
[07:46:35] Fold 1/5 | sparse matrices ready in 0.5s (total features: 110,000).
[07:46:36] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[07:46:45] Fold 1/5 | SGD Logistic Regression done in 9.4s (epochs=18, convergence warnings=0).
[07:46:45] Fold 1/5 | scoring held-out projects...
[07:46:45] Fold 1/5 | scoring done in 0.1s.
[07:46:46] Fold 1/5 completed in 6.24 min.
[07:46:46] Fold 2

## 18. Reload and validate saved inner-CV artifacts

In [22]:
import json
import pandas as pd

INNER_EXPERIMENT_NAME = exp0_config.experiment_name

INNER_OOF_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_oof_predictions.parquet"
)

INNER_POOLED_METRICS_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_pooled_metrics.json"
)

INNER_FOLD_METRICS_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_fold_metrics.csv"
)

INNER_FOLD_SUMMARY_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_fold_summary.csv"
)

INNER_THRESHOLD_METRICS_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_threshold_metrics.csv"
)

INNER_TOP_FEATURES_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_top_features.csv"
)

INNER_TRAINING_PATH = (
    INNER_CV_OUTPUT_DIR
    / f"{INNER_EXPERIMENT_NAME}_fold_training.csv"
)

inner_required_paths = [
    INNER_OOF_PATH,
    INNER_POOLED_METRICS_PATH,
    INNER_FOLD_METRICS_PATH,
    INNER_FOLD_SUMMARY_PATH,
    INNER_THRESHOLD_METRICS_PATH,
    INNER_TOP_FEATURES_PATH,
    INNER_TRAINING_PATH,
]

missing_inner_artifacts = [
    path for path in inner_required_paths if not path.exists()
]

if missing_inner_artifacts:
    print("Inner-CV artifacts are not available yet.")
    print("Run Section 17 first. Missing files:")
    for path in missing_inner_artifacts:
        print(" -", path.name)
else:
    inner_oof_df = pd.read_parquet(INNER_OOF_PATH)
    inner_fold_metrics_df = pd.read_csv(INNER_FOLD_METRICS_PATH)
    inner_fold_summary_df = pd.read_csv(INNER_FOLD_SUMMARY_PATH)
    inner_threshold_df = pd.read_csv(INNER_THRESHOLD_METRICS_PATH)
    inner_top_features_df = pd.read_csv(INNER_TOP_FEATURES_PATH)
    inner_training_df = pd.read_csv(INNER_TRAINING_PATH)

    with INNER_POOLED_METRICS_PATH.open("r", encoding="utf-8") as file:
        inner_pooled_metrics = json.load(file)

    assert len(inner_oof_df) == len(dev_df)
    assert inner_oof_df["source_row_id"].nunique() == len(dev_df)
    assert set(inner_oof_df["source_row_id"]) == set(dev_df["source_row_id"])
    assert set(inner_oof_df["source_row_id"]).isdisjoint(
        set(holdout_df["source_row_id"])
    )
    assert set(inner_oof_df["fold"].unique()) == set(range(INNER_N_SPLITS))
    assert inner_oof_df["y_score"].between(0.0, 1.0).all()
    assert set(inner_oof_df["label"].unique()) == {0, 1}

    print("✅ Inner OOF predictions cover every development row exactly once.")
    print("✅ No outer-holdout row appears in inner OOF predictions.")
    print("✅ Saved inner-CV artifacts loaded successfully.")

✅ Inner OOF predictions cover every development row exactly once.
✅ No outer-holdout row appears in inner OOF predictions.
✅ Saved inner-CV artifacts loaded successfully.


## 19. Inspect the inner-CV result — development data only

In [23]:
if "inner_pooled_metrics" not in globals():
    print("Run Sections 17 and 18 first.")
else:
    print(
        evaluation.format_metric_report(
            inner_pooled_metrics
        )
    )

    random_pr_baseline = inner_pooled_metrics["positive_rate"]
    model_pr_auc = inner_pooled_metrics["average_precision_pr_auc"]

    print("\nRanking lift over the development-set prevalence baseline:")
    print(f"Random PR baseline: {random_pr_baseline:.6f}")
    print(f"Model PR-AUC:       {model_pr_auc:.6f}")
    print(f"Ranking lift:       {model_pr_auc / random_pr_baseline:.2f}x")

    display(
        inner_fold_metrics_df[
            [
                "fold",
                "n_samples",
                "test_unique_projects",
                "positive_rate",
                "average_precision_pr_auc",
                "precision",
                "recall",
                "f1",
                "mcc",
                "false_positive_rate",
            ]
        ]
    )

    display(
        inner_training_df[
            [
                "fold",
                "train_rows",
                "test_rows",
                "word_tfidf_seconds",
                "char_tfidf_seconds",
                "model_fit_seconds",
                "total_fold_seconds",
                "model_n_iter",
                "convergence_warning_count",
            ]
        ]
    )

Pooled Out-of-Fold Evaluation
                   n_samples: 203958
                vulnerable_1: 10727
            non_vulnerable_0: 193231
               positive_rate: 0.052594
                   threshold: 0.500000
    average_precision_pr_auc: 0.125205
                   precision: 0.131065
                      recall: 0.377459
                          f1: 0.194570
                         mcc: 0.148525
                 specificity: 0.861078
         false_positive_rate: 0.138922
               true_negative: 166387
              false_positive: 26844
              false_negative: 6678
               true_positive: 4049

Ranking lift over the development-set prevalence baseline:
Random PR baseline: 0.052594
Model PR-AUC:       0.125205
Ranking lift:       2.38x


,fold,n_samples,test_unique_projects,positive_rate,average_precision_pr_auc,precision,recall,f1,mcc,false_positive_rate
0,0,55509,1,0.043597,0.088308,0.107271,0.271901,0.153846,0.109910,0.103148
1,1,40791,158,0.051261,0.135766,0.135294,0.417982,0.204421,0.165293,0.144341
2,2,39072,146,0.052774,0.118607,0.119123,0.382153,0.181630,0.133969,0.157444
3,3,35016,137,0.058459,0.140733,0.151872,0.390327,0.218664,0.167424,0.135339
4,4,33570,152,0.062764,0.149860,0.144792,0.441386,0.218054,0.164508,0.174586


,fold,train_rows,test_rows,word_tfidf_seconds,char_tfidf_seconds,model_fit_seconds,total_fold_seconds,model_n_iter,convergence_warning_count
0,0,148449,55509,133.380035,229.603552,9.386865,374.101307,18,0
1,1,163167,40791,125.852857,250.270173,11.577654,389.217749,19,0
2,2,164886,39072,126.072256,239.224428,11.445954,379.242075,20,0
3,3,168942,35016,132.601034,234.499717,8.757387,377.329757,14,0
4,4,170388,33570,129.904102,242.555538,13.734051,387.885425,20,0


## 20. Inspect stable inner-CV lexical signals

These are model diagnostics only. Coefficients show association in inner-CV
training folds; they are not causal explanations of vulnerability.

In [24]:
if "inner_top_features_df" not in globals():
    print("Run Sections 17 and 18 first.")
else:
    stable_inner_features = (
        inner_top_features_df
        .groupby(
            [
                "direction",
                "feature_type",
                "feature",
            ],
            as_index=False,
        )
        .agg(
            folds_present=("fold", "nunique"),
            mean_coefficient=("coefficient", "mean"),
            mean_abs_coefficient=("abs_coefficient", "mean"),
        )
        .sort_values(
            [
                "direction",
                "folds_present",
                "mean_abs_coefficient",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )
    )

    stable_vulnerable_words = stable_inner_features.loc[
        (stable_inner_features["direction"] == "vulnerable_associated")
        & (stable_inner_features["feature_type"] == "word")
        & (stable_inner_features["folds_present"] >= 3)
    ].head(30)

    display(stable_vulnerable_words)

    suspicious_terms = (
        "cve|vuln|vulnerab|security|overflow|exploit|attack|"
        "fix|patch|bug|unsafe|hack"
    )

    possible_label_hint_features = stable_inner_features.loc[
        stable_inner_features["feature"].str.contains(
            suspicious_terms,
            case=False,
            regex=True,
            na=False,
        )
    ].sort_values(
        [
            "folds_present",
            "mean_abs_coefficient",
        ],
        ascending=False,
    )

    print("Potential label-hint features:")
    display(possible_label_hint_features.head(50))

,direction,feature_type,feature,folds_present,mean_coefficient,mean_abs_coefficient
151,vulnerable_associated,word,word::open_flags,4,4.412359,4.412359
101,vulnerable_associated,word,word::RSocket,4,3.448480,3.448480
117,vulnerable_associated,word,word::context_handle,4,3.332741,3.332741
109,vulnerable_associated,word,word::ax25_dev,4,3.178083,3.178083
162,vulnerable_associated,word,word::sprintf buf,4,3.141850,3.141850
127,vulnerable_associated,word,word::duint32,4,3.112656,3.112656
116,vulnerable_associated,word,word::const QString,4,3.107809,3.107809
155,vulnerable_associated,word,word::quantum,4,2.963144,2.963144
136,vulnerable_associated,word,word::guc,4,2.819769,2.819769
166,vulnerable_associated,word,word::timr,4,2.775001,2.775001


Potential label-hint features:


,direction,feature_type,feature,folds_present,mean_coefficient,mean_abs_coefficient
53,non_vulnerable_associated,word,word::g_debug,1,-2.203334,2.203334


## 21. Model-selection checkpoint

At this point, EXP-0 has inner-CV evidence only. Do **not** run the outer
holdout yet if you intend to compare EXP-0 against EXP-1, EXP-2, or an
advanced model.

Choose the final candidate using only development-partition evidence such as
pooled inner-CV PR-AUC and operational considerations. Record the choice before
unlocking the final holdout.

In [25]:
FINAL_MODEL_LOCKED = False
FINAL_SELECTED_MODEL = None

if FINAL_MODEL_LOCKED:
    print("Final model is locked:", FINAL_SELECTED_MODEL)
else:
    print(
        "Final holdout remains protected. "
        "Run candidate experiments on dev first, then record one selected model."
    )

Final holdout remains protected. Run candidate experiments on dev first, then record one selected model.


## 22. Final outer-holdout evaluation — locked until selection is complete

Set both variables below only after model selection is complete:

```python
FINAL_MODEL_LOCKED = True
FINAL_SELECTED_MODEL = "cs1_exp0_lr_inner_dev_grouped"
```

This cell trains one EXP-0 model on all development projects and evaluates the
outer holdout once. Do not rerun it with different settings.

In [ ]:
# Deliberately separate from the inner-CV configuration/result.
FINAL_MODEL_LOCKED = False
FINAL_SELECTED_MODEL = None

if (
    FINAL_MODEL_LOCKED
    and FINAL_SELECTED_MODEL == exp0_config.experiment_name
):
    final_existing_files = list(FINAL_HOLDOUT_OUTPUT_DIR.iterdir())

    if final_existing_files:
        raise RuntimeError(
            "Final-holdout output directory is not empty. "
            "The holdout should be evaluated only once for the selected model."
        )

    exp0_final_holdout_results = exp0_lr.run_exp0_final_holdout(
        dev_frame=dev_df,
        holdout_frame=holdout_df,
        config=exp0_config,
        output_dir=FINAL_HOLDOUT_OUTPUT_DIR,
        additional_metadata={
            "evaluation_stage": "single_final_outer_holdout",
            "outer_holdout_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "selected_model_record": FINAL_SELECTED_MODEL,
            "selection_source": (
                "Development-only inner 5-fold grouped CV. "
                "The outer holdout was not used for model, threshold, or "
                "hyperparameter selection."
            ),
        },
    )

    print("FINAL OUTER-HOLDOUT METRICS")
    print("=" * 72)

    for metric_name, metric_value in (
        exp0_final_holdout_results["metrics"].items()
    ):
        print(f"{metric_name:>32}: {metric_value}")
else:
    print(
        "Final holdout remains locked. This is correct until the final model "
        "has been selected using development-only evidence."
    )

## 23. Final report-summary export

In [ ]:
if "inner_pooled_metrics" not in globals():
    print("Run Sections 17 and 18 first.")
else:
    inner_report_summary = pd.DataFrame(
        [
            {
                "experiment": "CS1-EXP0-LR",
                "evaluation_stage": "inner_development_cv",
                "model": (
                    "Class-weighted L2 Logistic Regression "
                    "optimized with SGD"
                ),
                "features": (
                    "Word TF-IDF (1,3) + Character TF-IDF (3,4)"
                ),
                "inner_validation": (
                    "5-fold StratifiedGroupKFold by project "
                    "on development partition only"
                ),
                "outer_holdout": (
                    "Project-disjoint outer partition reserved; "
                    "not evaluated in this notebook stage"
                ),
                "dev_samples": len(dev_df),
                "outer_holdout_samples": len(holdout_df),
                "inner_pr_auc": inner_pooled_metrics[
                    "average_precision_pr_auc"
                ],
                "inner_precision_at_050": inner_pooled_metrics[
                    "precision"
                ],
                "inner_recall_at_050": inner_pooled_metrics[
                    "recall"
                ],
                "inner_f1_at_050": inner_pooled_metrics[
                    "f1"
                ],
                "inner_mcc_at_050": inner_pooled_metrics[
                    "mcc"
                ],
            }
        ]
    )

    INNER_REPORT_SUMMARY_PATH = (
        INNER_CV_OUTPUT_DIR / "cs1_exp0_inner_dev_report_summary.csv"
    )

    inner_report_summary.to_csv(
        INNER_REPORT_SUMMARY_PATH,
        index=False,
    )

    display(inner_report_summary)
    print("Saved inner-CV report summary:", INNER_REPORT_SUMMARY_PATH)